# Khám phá dữ liệu tin tức

Khảo sát toàn bộ `data/raw/news_raw.csv` để xác định những vấn đề cần xử lý trước khi làm sạch.

Notebook này không lọc ngày, loại QNS hoặc sửa dữ liệu raw. Kết quả được lưu trong `data/interim` để đối chiếu ở bước clean.


## 1. Đọc dữ liệu


In [1]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

pd.set_option('display.max_columns', 30)
pd.set_option('display.max_colwidth', 80)


In [2]:
CURRENT_DIR = Path.cwd().resolve()
PROJECT_CANDIDATES = [
    CURRENT_DIR / 'news_processing', CURRENT_DIR, *CURRENT_DIR.parents,
]
PROJECT_DIR = next(
    (directory for directory in PROJECT_CANDIDATES
     if (directory / 'data' / 'raw' / 'news_raw.csv').is_file()
     and (directory / 'src').is_dir()),
    None,
)
if PROJECT_DIR is None:
    raise FileNotFoundError('Không xác định được thư mục news_processing.')

RAW_FILE = PROJECT_DIR / 'data' / 'raw' / 'news_raw.csv'
INTERIM_DIR = PROJECT_DIR / 'data' / 'interim'
sys.path.insert(0, str(PROJECT_DIR))


In [3]:
from src.cleaning_utils import parse_serialized_list, select_scalar_value
from src.validation_utils import (
    canonicalize_url,
    classify_timestamp_format,
    is_valid_http_url,
    normalize_title,
    parse_timestamp,
    split_tickers,
    summarize_duplicates,
)


In [4]:
if not RAW_FILE.exists():
    raise FileNotFoundError(f'Không tìm thấy file: {RAW_FILE}')

raw_news = pd.read_csv(RAW_FILE, low_memory=False)
print(f'Số bài: {len(raw_news):,}')
print(f'Số trường: {raw_news.shape[1]}')


Số bài: 4,491
Số trường: 22


## 2. Cấu trúc và giá trị thiếu


In [5]:
EXPECTED_RAW_COLUMNS = [
    'news_id', 'source', 'url', 'title', 'summary', 'content',
    'published_date', 'published_at_raw', 'author', 'category',
    'crawl_time', 'crawl_status', 'content_hash', 'industry_group',
    'tickers', 'keywords', 'event_type', 'note', 'dataset_origin',
    'source_news_ids', 'source_content_hashes', 'original_crawl_status',
]
schema_report = pd.DataFrame([{
    'expected': len(EXPECTED_RAW_COLUMNS),
    'actual': raw_news.shape[1],
    'missing': sorted(set(EXPECTED_RAW_COLUMNS) - set(raw_news.columns)),
    'extra': sorted(set(raw_news.columns) - set(EXPECTED_RAW_COLUMNS)),
}])
display(schema_report)


,expected,actual,missing,extra
0,22,22,[],[]


In [6]:
structure_report = pd.DataFrame({
    'column': raw_news.columns,
    'dtype': raw_news.dtypes.astype(str).to_numpy(),
    'missing_count': raw_news.isna().sum().to_numpy(),
    'missing_percent': (raw_news.isna().mean() * 100).round(2).to_numpy(),
    'unique_count': raw_news.nunique(dropna=True).to_numpy(),
})
display(structure_report)


,column,dtype,missing_count,missing_percent,unique_count
0,news_id,object,0,0.00,4491
1,source,object,0,0.00,37
2,url,object,0,0.00,4491
3,title,object,0,0.00,4237
4,summary,object,3,0.07,4210
5,content,object,0,0.00,4485
6,published_date,object,0,0.00,1548
7,published_at_raw,object,858,19.10,3498
8,author,object,934,20.80,260
9,category,object,2648,58.96,100


## 3. Thời gian phát hành và thu thập

Ngày đăng và ngày crawl có ý nghĩa khác nhau. Các thời điểm có múi giờ được quy về giờ Việt Nam trước khi so sánh ngày.


In [7]:
published_dates = pd.to_datetime(
    raw_news['published_date'], format='%Y-%m-%d', errors='coerce'
)
source_timestamps = raw_news['published_at_raw'].map(parse_timestamp)
crawl_timestamps = raw_news['crawl_time'].map(parse_timestamp)


In [8]:
time_summary = pd.DataFrame([
    {'field': 'published_date', 'valid': published_dates.notna().sum(),
     'missing': published_dates.isna().sum(),
     'min': str(published_dates.min()), 'max': str(published_dates.max())},
    {'field': 'published_at_raw', 'valid': source_timestamps.notna().sum(),
     'missing': source_timestamps.isna().sum(),
     'min': str(source_timestamps.min()), 'max': str(source_timestamps.max())},
    {'field': 'crawl_time', 'valid': crawl_timestamps.notna().sum(),
     'missing': crawl_timestamps.isna().sum(),
     'min': str(crawl_timestamps.min()), 'max': str(crawl_timestamps.max())},
])
display(time_summary)


,field,valid,missing,min,max
0,published_date,4491,0,2020-01-01 00:00:00,2026-09-25 00:00:00
1,published_at_raw,3633,858,2020-01-01 10:35:58+07:00,2026-09-25 21:42:25+07:00
2,crawl_time,4491,0,2026-09-28 13:21:18+07:00,2026-09-29 12:12:49+07:00


Ngày phát hành trải từ 01/01/2020 đến 25/09/2026, trong khi dữ liệu được crawl ngày 28–29/09/2026. Vì vậy, không dùng `crawl_time` thay cho ngày đăng bài.


### 3.1. Đối chiếu ngày phát hành

So sánh `published_date` với ngày trong `published_at_raw` để phát hiện sai lệch trước khi lọc thời gian. Định dạng không nhận diện được sẽ trả về giá trị thiếu, không đoán ngày.


In [9]:
timestamp_formats = raw_news['published_at_raw'].map(classify_timestamp_format)
source_dates = source_timestamps.dt.tz_localize(None).dt.normalize()
comparable_date_mask = published_dates.notna() & source_dates.notna()
date_conflict_mask = comparable_date_mask & published_dates.ne(source_dates)

date_conflict_candidates = raw_news.loc[date_conflict_mask, [
    'news_id', 'source', 'url', 'title', 'published_date',
    'published_at_raw', 'crawl_time',
]].copy()
date_conflict_candidates['published_at_local_date'] = (
    source_dates.loc[date_conflict_mask].dt.strftime('%Y-%m-%d').to_numpy()
)

print('Timestamp unknown:', int(timestamp_formats.eq('unknown').sum()))
print('Dòng có thể đối chiếu:', int(comparable_date_mask.sum()))
print('Dòng xung đột ngày:', int(date_conflict_mask.sum()))
display(date_conflict_candidates.head(5))


Timestamp unknown: 0
Dòng có thể đối chiếu: 3633
Dòng xung đột ngày: 1116


,news_id,source,url,title,published_date,published_at_raw,crawl_time,published_at_local_date
3,d5613237f2ce9178c690cf55b0bb86f7a338e07f24421da1fe7158a201fb4d02,BNews,https://bnews.vn/vinamilk-ho-tro-1-ty-dong-cho-cac-y-bac-sy-va-benh-nhan-ben...,Vinamilk hỗ trợ 1 tỷ đồng cho các y bác sỹ và bệnh nhân Bệnh viện Bạch Mai,2020-01-04,2020-04-01T09:32:51+07:00,2026-09-28T11:23:38Z,2020-04-01
4,ed810beeb41d915a5163400d04801c80f7a8ab8cbaff57ea0609e32f94ab9fdf,VietnamNet,https://vietnamnet.vn/vinamilk-tang-bv-bach-mai-150-nghin-san-pham-sua-dinh-...,Vinamilk tặng BV Bạch Mai 150 nghìn sản phẩm sữa dinh dưỡng,2020-01-04,2020-04-01T11:00:00.000 +07:00,2026-09-28T10:15:00Z,2020-04-01
5,992f45bd84303184c1382b76db1d02396f6d266eb9530c70893aaca1bcfb223f,BNews,https://bnews.vn/vinamilk-tiep-can-cac-thi-truong-xuat-khau-moi/161350.html,Vinamilk tiếp cận các thị trường xuất khẩu mới,2020-01-07,2020-07-01T12:36:39+07:00,2026-09-28T11:23:38Z,2020-07-01
6,ad428f7639cf04076a2b53eae774407f3659a0c3ade14e1187bf6fdf93755fcd,BNews,https://bnews.vn/pnj-ung-ho-9-500-ong-sinh-thiet-xet-nghiem-covid-19-cho-qua...,PNJ ủng hộ 9.500 ống sinh thiết xét nghiệm COVID-19 cho Quảng Ngãi,2020-01-09,2020-09-01T19:46:27+07:00,2026-09-28T11:23:38Z,2020-09-01
7,40957692491ce4b6b3eeca5ff92190ce649479991435388acd101d1f014b7e06,BNews,https://bnews.vn/vinamilk-thuong-hieu-manh-thuoc-top-1-000-thuong-hieu-hang-...,Vinamilk thương hiệu mạnh thuộc TOP 1.000 thương hiệu hàng đầu châu Á,2020-01-10,2020-10-01T15:43:40+07:00,2026-09-28T11:23:38Z,2020-10-01


Có 1.116 trong 3.633 dòng đối chiếu được có ngày không thống nhất. Cần xử lý sự khác biệt này ở bước clean trước khi áp dụng khoảng thời gian nghiên cứu.


## 4. Nguồn và mã cổ phiếu


In [10]:
source_distribution = (
    raw_news['source'].value_counts()
    .rename_axis('source')
    .reset_index(name='article_count')
)
print('Số nguồn:', raw_news['source'].nunique())
display(source_distribution.head(10))


Số nguồn: 37


,source,article_count
0,CafeF,1267
1,API:CafeF DN,840
2,VietnamNet,751
3,BNews,721
4,CafeBiz,247
5,MWGOfficial,146
6,VietnamFinance,60
7,NguoiQuanSat,50
8,TinNhanhChungKhoan,48
9,NhaDauTu,44


### 4.1. Phân bố mã cổ phiếu

Một bài có thể chứa nhiều mã, nên tổng lượt xuất hiện theo ticker không bằng số bài.


In [11]:
ticker_lists = raw_news['tickers'].map(split_tickers)
ticker_distribution = (
    ticker_lists.explode().value_counts()
    .rename_axis('ticker')
    .reset_index(name='article_count')
)
display(ticker_distribution)


,ticker,article_count
0,MWG,930
1,MSN,769
2,VNM,736
3,PNJ,717
4,FRT,449
5,SAB,269
6,DGW,196
7,ANV,191
8,DBC,155
9,DHG,97


### 4.2. Các bài chứa QNS


In [12]:
ticker_sets = ticker_lists.map(set)
qns_any_mask = ticker_sets.map(lambda values: 'QNS' in values)
qns_only_mask = ticker_sets.map(lambda values: values == {'QNS'})
qns_mixed_mask = ticker_sets.map(
    lambda values: 'QNS' in values and bool(values - {'QNS'})
)
qns_summary = pd.DataFrame([{
    'contains_QNS': int(qns_any_mask.sum()),
    'QNS_only': int(qns_only_mask.sum()),
    'QNS_with_other': int(qns_mixed_mask.sum()),
}])
display(qns_summary)


,contains_QNS,QNS_only,QNS_with_other
0,64,56,8


Với 8 bài có QNS cùng mã khác, bước clean chỉ gỡ QNS khỏi danh sách mã. Các bài chỉ có QNS mới bị loại.


## 5. Các vấn đề cần xử lý

### 5.1. Danh sách bị lưu thành chuỗi

Một số ô vốn chỉ cần một giá trị lại chứa chuỗi dạng `['giá trị']`. Hàm kiểm tra dùng `ast.literal_eval` để nhận diện danh sách hợp lệ, tránh coi tiêu đề bắt đầu bằng `[PDF]` là lỗi.


In [13]:
SCALAR_COLUMNS = ['url', 'title', 'summary', 'author', 'category']
serialized_list_masks = pd.DataFrame({
    column: raw_news[column].map(parse_serialized_list).notna()
    for column in SCALAR_COLUMNS
})
serialized_row_mask = serialized_list_masks.any(axis=1)
bracket_non_list_mask = (
    raw_news['title'].astype(str).str.strip().str.startswith('[')
    & ~serialized_list_masks['title']
)
display(serialized_list_masks.sum().to_frame('row_count'))
print('Dòng có list thực sự:', int(serialized_row_mask.sum()))
print('Tiêu đề [ nhưng không phải list:', int(bracket_non_list_mask.sum()))


,row_count
url,31
title,31
summary,31
author,25
category,25


Dòng có list thực sự: 31
Tiêu đề [ nhưng không phải list: 3


### 5.2. URL trùng

Bỏ tiền tố `www`/`m`, query, fragment và dấu `/` cuối để so sánh URL. Phép chuẩn hóa này chỉ tạo giá trị đối chiếu, không thay đổi dữ liệu raw.


In [14]:
raw_url_valid_mask = raw_news['url'].map(is_valid_http_url)
scalar_urls = raw_news['url'].map(select_scalar_value)
canonical_urls = scalar_urls.map(canonicalize_url)
duplicate_url_mask = canonical_urls.duplicated(keep=False) & canonical_urls.ne('')
url_report = pd.DataFrame([{
    'invalid_raw_url': int((~raw_url_valid_mask).sum()),
    'invalid_after_list_parse': int((~scalar_urls.map(is_valid_http_url)).sum()),
    'duplicate_rows': int(duplicate_url_mask.sum()),
    'duplicate_groups': int(canonical_urls[duplicate_url_mask].nunique()),
}])
display(url_report)


,invalid_raw_url,invalid_after_list_parse,duplicate_rows,duplicate_groups
0,31,0,138,69


31 URL không hợp lệ ở dạng raw đều đọc được sau khi tách chuỗi list; đây không phải 31 liên kết hỏng.

### 5.3. Trùng tiêu đề và nội dung

Chuẩn hóa Unicode, chữ hoa/thường và khoảng trắng để đối chiếu tiêu đề. Nội dung được so sánh nguyên văn.


In [15]:
scalar_titles = raw_news['title'].map(select_scalar_value).fillna('')
normalized_titles = scalar_titles.map(normalize_title)
duplicate_title_mask = normalized_titles.duplicated(keep=False) & normalized_titles.ne('')
duplicate_content_mask = (
    raw_news['content'].duplicated(keep=False) & raw_news['content'].notna()
)
duplicate_candidate_mask = (
    duplicate_url_mask | duplicate_title_mask | duplicate_content_mask
)
duplicate_summary = pd.DataFrame([
    summarize_duplicates(canonical_urls, 'canonical_url'),
    summarize_duplicates(normalized_titles, 'normalized_title'),
    summarize_duplicates(raw_news['content'].fillna(''), 'exact_content'),
])
display(duplicate_summary)
print('news_id trùng:', int(raw_news['news_id'].duplicated(keep=False).sum()))
print('Tổng ứng viên review:', int(duplicate_candidate_mask.sum()))


,check,rows_in_duplicate_groups,duplicate_groups,extra_rows
0,canonical_url,138,69,69
1,normalized_title,559,271,288
2,exact_content,10,4,6


news_id trùng: 0
Tổng ứng viên review: 565


In [16]:
duplicate_candidates = raw_news.loc[duplicate_candidate_mask, [
    'news_id', 'source', 'url', 'title', 'published_date', 'tickers',
]].copy()
duplicate_candidates['canonical_url'] = canonical_urls.loc[duplicate_candidate_mask]
duplicate_candidates['duplicate_url'] = duplicate_url_mask.loc[duplicate_candidate_mask]
duplicate_candidates['duplicate_title'] = duplicate_title_mask.loc[duplicate_candidate_mask]
duplicate_candidates['duplicate_content'] = duplicate_content_mask.loc[duplicate_candidate_mask]
display(duplicate_candidates.head(10))


,news_id,source,url,title,published_date,tickers,canonical_url,duplicate_url,duplicate_title,duplicate_content
135,73bbf4da7a89ea374f5b37c713da7d1ed10673f678ba58d94785995fddd75885,VietnamNet,https://vietnamnet.vn/hoi-nghi-va-trien-lam-the-gioi-so-2020-se-chu-trong-va...,Hội nghị và Triển lãm Thế giới Số 2020 sẽ chú trọng vào chuyển đổi số,2020-08-01,DGW,https://vietnamnet.vn/hoi-nghi-va-trien-lam-the-gioi-so-2020-se-chu-trong-va...,False,True,False
136,d07a4baae675ba59d8dbdcf648edc7e8b35c7423ac8272885c96ec959e9dc4e0,VietnamNet,https://vietnamnet.vn/hoi-nghi-va-trien-lam-the-gioi-so-2020-se-chu-trong-va...,Hội nghị và Triển lãm Thế giới Số 2020 sẽ chú trọng vào chuyển đổi số,2020-08-01,DGW,https://vietnamnet.vn/hoi-nghi-va-trien-lam-the-gioi-so-2020-se-chu-trong-va...,False,True,False
155,a6b55d10a45686f71572a7b2f631f8ece6801e4e4ecec1b9cfe878a0fa7592ca,VietnamNet,https://vietnamnet.vn/trien-lam-the-gioi-so-2020-se-khoi-dau-chang-duong-pha...,Triển lãm Thế giới số 2020 sẽ khởi đầu chặng đường phát triển mới của ICT to...,2020-09-10,DGW,https://vietnamnet.vn/trien-lam-the-gioi-so-2020-se-khoi-dau-chang-duong-pha...,False,True,False
156,75b21c21c69fc2acdea909cfdae5aa635fe97f28535e41d6a274084f0205725f,VietnamNet,https://vietnamnet.vn/trien-lam-the-gioi-so-2020-se-khoi-dau-chang-duong-pha...,Triển lãm Thế giới số 2020 sẽ khởi đầu chặng đường phát triển mới của ICT to...,2020-09-10,DGW,https://vietnamnet.vn/trien-lam-the-gioi-so-2020-se-khoi-dau-chang-duong-pha...,False,True,False
162,8b46a1779d8c00f060b77928e00775f9e7ada4417ef2aedda3a2140e02d386d1,VietnamNet,https://vietnamnet.vn/trien-lam-the-gioi-so-2020-se-to-chuc-truc-tuyen-tren-...,Triển lãm Thế giới số 2020 sẽ tổ chức trực tuyến trên nền tảng 'Make in Viet...,2020-09-25,DGW,https://vietnamnet.vn/trien-lam-the-gioi-so-2020-se-to-chuc-truc-tuyen-tren-...,False,True,False
163,500139748121eda45b2401cc6a10bb0d9a4cbd009a5884f234afac0f8cddbb49,VietnamNet,https://vietnamnet.vn/trien-lam-the-gioi-so-2020-se-to-chuc-truc-tuyen-tren-...,Triển lãm Thế giới số 2020 sẽ tổ chức trực tuyến trên nền tảng 'Make in Viet...,2020-09-25,DGW,https://vietnamnet.vn/trien-lam-the-gioi-so-2020-se-to-chuc-truc-tuyen-tren-...,False,True,False
189,5bbd21cd14c072c7c82a27fea35ad3f250be95925bdaede9aadcd3eecac4d3d0,BNews,https://bnews.vn/khai-mac-hoi-nghi-va-trien-lam-truc-tuyen-the-gioi-so-2020/...,Khai mạc Hội nghị và Triển lãm trực tuyến Thế giới số 2020,2020-10-20,DGW,https://bnews.vn/khai-mac-hoi-nghi-va-trien-lam-truc-tuyen-the-gioi-so-2020/...,False,True,False
190,ed2879230068ebdf3d6805e4884610e11dc1a68b5bfbb6f35d834815a58a416f,VietnamNet,https://vietnamnet.vn/khai-mac-hoi-nghi-va-trien-lam-truc-tuyen-the-gioi-so-...,Khai mạc Hội nghị và Triển lãm trực tuyến Thế giới số 2020,2020-10-20,DGW,https://vietnamnet.vn/khai-mac-hoi-nghi-va-trien-lam-truc-tuyen-the-gioi-so-...,False,True,False
192,6f37fa70bcb8d81ec1e3c3b5d4ab97edcc42df3ab2d372e13da2902482c2bdca,VietnamNet,https://vietnamnet.vn/hoi-nghi-bo-truong-itu-hop-tac-thuc-hien-su-menh-cung-...,Hội nghị Bộ trưởng ITU: Hợp tác thực hiện sứ mệnh 'cùng nhau xây dựng thế gi...,2020-10-21,DGW,https://vietnamnet.vn/hoi-nghi-bo-truong-itu-hop-tac-thuc-hien-su-menh-cung-...,False,True,False
193,dc2f030b94c3938ec4a3b1ba0b6a7c943617609b1bc29e7e3a98df8b179e0b37,VietnamNet,https://vietnamnet.vn/hoi-nghi-bo-truong-itu-hop-tac-thuc-hien-su-menh-cung-...,Hội nghị Bộ trưởng ITU: Hợp tác thực hiện sứ mệnh 'cùng nhau xây dựng thế gi...,2020-10-21,DGW,https://vietnamnet.vn/hoi-nghi-bo-truong-itu-hop-tac-thuc-hien-su-menh-cung-...,False,True,False


565 dòng được đánh dấu là ứng viên trùng, chưa phải 565 bài cần xóa. Trùng tiêu đề chưa đủ để kết luận hai bài có cùng nội dung.

### 5.4. Độ dài văn bản


In [17]:
text_values = {
    'title': scalar_titles,
    'summary': raw_news['summary'].map(select_scalar_value),
    'content': raw_news['content'],
}
text_quality_rows = []
for field, values in text_values.items():
    lengths = values.fillna('').astype(str).str.strip().str.len()
    non_empty = lengths[lengths.gt(0)]
    text_quality_rows.append({
        'field': field, 'missing_or_blank': int(lengths.eq(0).sum()),
        'min_non_empty': int(non_empty.min()),
        'median_length': round(float(non_empty.median()), 1),
        'under_30_chars': int(lengths.lt(30).sum()),
        'under_200_chars': int(lengths.lt(200).sum()),
    })
text_quality_report = pd.DataFrame(text_quality_rows)
display(text_quality_report)


,field,missing_or_blank,min_non_empty,median_length,under_30_chars,under_200_chars
0,title,0,9,72.0,58,4490
1,summary,3,2,165.0,11,3407
2,content,0,214,4138.0,0,0


Không có tiêu đề hoặc nội dung rỗng. Tuy nhiên, độ dài không chứng minh nội dung đầy đủ hay đã sạch boilerplate; phần này cần được kiểm tra ở bước clean.


## 6. Lưu kết quả khảo sát

Lưu các bảng đối chiếu để truy lại những dòng cần xử lý. File `invalid_records.csv` chứa các ứng viên cần kiểm tra, không phải danh sách bài tự động bị loại.


In [18]:
issue_flags = pd.DataFrame({
    'date_conflict': date_conflict_mask,
    'serialized_list': serialized_row_mask,
    'invalid_raw_url': ~raw_url_valid_mask,
    'duplicate_canonical_url': duplicate_url_mask,
    'duplicate_normalized_title': duplicate_title_mask,
    'duplicate_exact_content': duplicate_content_mask,
})
review_row_mask = issue_flags.any(axis=1)
review_candidates = raw_news.loc[review_row_mask, [
    'news_id', 'source', 'url', 'title', 'published_date', 'tickers',
]].copy()
review_candidates = review_candidates.join(issue_flags.loc[review_row_mask])


In [19]:
validation_summary = pd.DataFrame([
    {'metric': 'eda_status', 'value': 'REQUIRES_CLEANING'},
    {'metric': 'total_rows', 'value': len(raw_news)},
    {'metric': 'total_columns', 'value': raw_news.shape[1]},
    {'metric': 'published_at_missing', 'value': int(source_timestamps.isna().sum())},
    {'metric': 'date_conflicts', 'value': int(date_conflict_mask.sum())},
    {'metric': 'serialized_list_rows', 'value': int(serialized_row_mask.sum())},
    {'metric': 'invalid_raw_urls', 'value': int((~raw_url_valid_mask).sum())},
    {'metric': 'duplicate_canonical_url_rows', 'value': int(duplicate_url_mask.sum())},
    {'metric': 'duplicate_title_candidates', 'value': int(duplicate_title_mask.sum())},
    {'metric': 'duplicate_content_candidates', 'value': int(duplicate_content_mask.sum())},
    {'metric': 'review_candidate_rows', 'value': int(review_row_mask.sum())},
    {'metric': 'qns_total_raw', 'value': int(qns_any_mask.sum())},
    {'metric': 'qns_only_raw', 'value': int(qns_only_mask.sum())},
    {'metric': 'qns_mixed_raw', 'value': int(qns_mixed_mask.sum())},
])
display(validation_summary)


,metric,value
0,eda_status,REQUIRES_CLEANING
1,total_rows,4491
2,total_columns,22
3,published_at_missing,858
4,date_conflicts,1116
5,serialized_list_rows,31
6,invalid_raw_urls,31
7,duplicate_canonical_url_rows,138
8,duplicate_title_candidates,559
9,duplicate_content_candidates,10


In [20]:
INTERIM_DIR.mkdir(parents=True, exist_ok=True)
output_tables = {
    INTERIM_DIR / 'raw_data_structure.csv': structure_report,
    INTERIM_DIR / 'validation_summary.csv': validation_summary,
    INTERIM_DIR / 'date_conflict_candidates.csv': date_conflict_candidates,
    INTERIM_DIR / 'duplicate_candidates.csv': duplicate_candidates,
    INTERIM_DIR / 'invalid_records.csv': review_candidates,
}
for output_path, output_table in output_tables.items():
    output_table.to_csv(output_path, index=False, encoding='utf-8-sig')


In [21]:
output_report = pd.DataFrame([
    {'file': path.relative_to(PROJECT_DIR).as_posix(), 'rows': len(table)}
    for path, table in output_tables.items()
])
display(output_report)


,file,rows
0,data/interim/raw_data_structure.csv,22
1,data/interim/validation_summary.csv,14
2,data/interim/date_conflict_candidates.csv,1116
3,data/interim/duplicate_candidates.csv,565
4,data/interim/invalid_records.csv,1504


## 7. Kết luận

Dữ liệu cần xử lý sai lệch ngày, các ô lưu dạng list và bài trùng trước khi dùng cho NLP. Việc lọc khoảng 19/10/2021–25/09/2026 và loại QNS được thực hiện trong `02_clean_news.ipynb`, không áp dụng ở EDA.
